# Fine-tuning نهایی ParsBERT برای تحلیل نظرهای دیجی‌کالا

این نوت‌بوک برای Google Colab و GPU (از جمله Tesla T4) آماده شده است. مدل برندهٔ مسابقه، مدل ‎**HooshvareLab/bert-fa-base-uncased**، با تمام **3,572,367 ردیف train_supervised.csv** آموزش می‌بیند. ورودی فقط متن body است و دو خروجی recommendation_status و rating_band دارد. برای هر هدف، برچسب ناموجود فقط loss همان head را نادیده می‌گیرد. دادهٔ مصنوعاً بدون‌برچسب در این fine-tuning نظارت‌شده وارد نمی‌شود؛ آزمایش نیمه‌نظارتی جداست.

آموزش روی T4 زمان‌بر است و ممکن است چند نشست Colab طول بکشد. checkpoint بازیابی شامل وزن‌ها و optimizer/scheduler/scaler و نقطهٔ ادامه در Drive ذخیره می‌شود. اگر نشست قطع شد، runtime تازه بگیرید، سلول‌ها را از ابتدا اجرا کنید تا checkpoint بازیابی شود.

## نقشهٔ اجرا و اصول ارزیابی

- اتصال Drive و کنترل مسیر splitهای آماده.
- کنترل schema و manifest بدون بارگذاری کل CSV در حافظه.
- fine-tuning کامل ParsBERT با داده‌خوانی chunkی، mixed precision و gradient accumulation.
- پایش پس از هر epoch روی نمونهٔ ثابت 100,000 ردیفی validation؛ فقط برای پایش و توقف زودهنگام.
- انتخاب بهترین checkpoint با میانگین Macro F1 دو هدف.
- ارزیابی نهایی روی تمام validation و تمام test؛ test فقط یک‌بار در این مرحله استفاده می‌شود.
- ذخیرهٔ نمودارهای وضوح بالا، معیارها، tokenizer، وزن‌ها و تنظیمات در Drive.

گزارش نهایی باید نمونه‌ای بودن پایش میان‌دوره‌ای را از ارزیابی کامل جدا کند.

## 1. اتصال Drive و مسیرها

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, json, math, random, time, gc, platform, shutil
import numpy as np
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/digikala-transformer")
DATA_DIR = PROJECT_DIR / "data" / "clean_data" / "split_body_cleaned"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "parsbert_finetuning"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
PLOT_DIR = OUTPUT_DIR / "plots"
REPORT_DIR = OUTPUT_DIR / "reports"
CACHE_DIR = OUTPUT_DIR / "cache"
for p in [OUTPUT_DIR, CHECKPOINT_DIR, PLOT_DIR, REPORT_DIR, CACHE_DIR]: p.mkdir(parents=True, exist_ok=True)
TRAIN_PATH, VALID_PATH, TEST_PATH = [DATA_DIR / n for n in ["train_supervised.csv", "validation.csv", "test.csv"]]
MANIFEST_PATH = DATA_DIR / "split_manifest.json"
missing = [str(p) for p in [TRAIN_PATH, VALID_PATH, TEST_PATH, MANIFEST_PATH] if not p.exists()]
if missing: raise FileNotFoundError("مسیر را اصلاح کنید:\n" + "\n".join(missing))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

# Cache the large input CSVs on the VM disk; model outputs remain on Drive.
LOCAL_DATA_DIR = Path("/content/digikala_split_cache")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
_drive_paths = {"train_supervised.csv": TRAIN_PATH, "validation.csv": VALID_PATH, "test.csv": TEST_PATH, "split_manifest.json": MANIFEST_PATH}
for _name, _src in _drive_paths.items():
    _dst = LOCAL_DATA_DIR / _name
    if not _dst.exists() or _dst.stat().st_size != _src.stat().st_size:
        print("Copying to local Colab disk:", _name)
        shutil.copy2(_src, _dst)
TRAIN_PATH = LOCAL_DATA_DIR / "train_supervised.csv"
VALID_PATH = LOCAL_DATA_DIR / "validation.csv"
TEST_PATH = LOCAL_DATA_DIR / "test.csv"
MANIFEST_PATH = LOCAL_DATA_DIR / "split_manifest.json"
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
print("Train rows:", f"{manifest['output_row_counts']['train_supervised']:,}")
print("Validation:", f"{manifest['output_row_counts']['validation']:,}", "Test:", f"{manifest['output_row_counts']['test']:,}")

## 2. وابستگی‌ها و کنترل GPU

In [ ]:
%pip -q install "transformers>=4.44,<5" safetensors scikit-learn matplotlib seaborn tqdm

In [ ]:
import torch, transformers, sklearn, matplotlib, seaborn as sns
from IPython.display import display
from tqdm.auto import tqdm
if not torch.cuda.is_available():
    raise RuntimeError("GPU فعال نیست. در Colab از Runtime > Change runtime type گزینه GPU را انتخاب کنید.")
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"{GPU_NAME} | VRAM {VRAM_GB:.1f} GB | PyTorch {torch.__version__} | Transformers {transformers.__version__}")
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__)

## 3. تنظیمات آموزش

برای سازگاری بهتر با Colab رایگان، این اجرا دقیقاً ۲ میلیون ردیف را با نمونه‌گیری تصادفی، یکنواخت و ثابت از کل train انتخاب می‌کند؛ ابتدای CSV به‌تنهایی انتخاب نمی‌شود. seed و indexهای انتخاب‌شده ذخیره می‌شوند تا ادامهٔ نشست دقیقاً همان نمونه را ببیند. این یک زیرنمونهٔ بازتولیدپذیر از train است، نه آموزش روی تمام 3.57 میلیون ردیف.

تنظیمات T4: batch برابر 8 و accumulation برابر 4 (batch مؤثر 32)، mixed precision و gradient checkpointing. در صورت Out of Memory، batch را 4 و accumulation را 8 کنید. پس از 200 batch نرخ واقعی تخمین زده می‌شود؛ برآورد چاپ‌شده فقط زمان آموزش است و evaluation/checkpoint I/O را شامل نمی‌شود.

In [ ]:
SEED = 2026
CHECKPOINT = "HooshvareLab/bert-fa-base-uncased"
TEXT_COLUMN = "body"
TARGETS = {"recommendation_status": ["recommended", "not_recommended", "no_idea"],
           "rating_band": ["low", "medium", "high"]}
MAX_LENGTH = 128
BATCH_SIZE = 8
GRAD_ACCUMULATION = 4
NUM_EPOCHS = 3
ENCODER_LR, HEAD_LR = 2e-5, 1e-4
WEIGHT_DECAY, WARMUP_RATIO, MAX_GRAD_NORM = 0.01, 0.06, 1.0
CHUNK_ROWS = 20_000
VALIDATION_MONITOR_SIZE, VALIDATION_MONITOR_SEED = 100_000, 2026
EARLY_STOPPING_PATIENCE = 2
SAVE_EVERY_OPTIMIZER_STEPS = 1_000
USE_GRADIENT_CHECKPOINTING = True
MAX_TRAIN_ROWS = 2_000_000  # دقیقاً دو میلیون ردیف با نمونه‌گیری تصادفی ثابت
TRAIN_SAMPLE_SEED = 2026

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
EXPECTED_TRAIN_ROWS = int(manifest["output_row_counts"]["train_supervised"])
TRAIN_ROWS = min(EXPECTED_TRAIN_ROWS, MAX_TRAIN_ROWS) if MAX_TRAIN_ROWS else EXPECTED_TRAIN_ROWS
TRAIN_INDEX_PATH = CACHE_DIR / f"train_uniform_indices_n{TRAIN_ROWS}_seed{TRAIN_SAMPLE_SEED}.npy"
if TRAIN_ROWS == EXPECTED_TRAIN_ROWS:
    TRAIN_ROW_INDICES = None
else:
    if TRAIN_INDEX_PATH.exists():
        TRAIN_ROW_INDICES = np.load(TRAIN_INDEX_PATH)
    else:
        TRAIN_ROW_INDICES = np.sort(np.random.default_rng(TRAIN_SAMPLE_SEED).choice(EXPECTED_TRAIN_ROWS, size=TRAIN_ROWS, replace=False))
        np.save(TRAIN_INDEX_PATH, TRAIN_ROW_INDICES)
    if len(TRAIN_ROW_INDICES) != TRAIN_ROWS or TRAIN_ROW_INDICES[-1] >= EXPECTED_TRAIN_ROWS:
        raise ValueError("فهرست نمونهٔ train با اندازهٔ فایل/تنظیم فعلی سازگار نیست؛ فایل npy cache را بررسی کنید.")
EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUMULATION
MICRO_BATCHES_PER_EPOCH = math.ceil(TRAIN_ROWS / BATCH_SIZE)
OPTIMIZER_STEPS_PER_EPOCH = math.ceil(MICRO_BATCHES_PER_EPOCH / GRAD_ACCUMULATION)
print(f"Rows: {TRAIN_ROWS:,}/{EXPECTED_TRAIN_ROWS:,} | effective batch: {EFFECTIVE_BATCH} | max length: {MAX_LENGTH}")
print(f"Optimizer steps per epoch: {OPTIMIZER_STEPS_PER_EPOCH:,} | epochs: {NUM_EPOCHS}")
if TRAIN_ROWS != EXPECTED_TRAIN_ROWS: print(f"Seeded uniform sample: {TRAIN_ROWS:,} rows | seed={TRAIN_SAMPLE_SEED}")

## 4. کنترل داده و manifest

فقط body و دو target به مدل داده می‌شوند. سایر ستون‌ها، از جمله rate و product_id، feature نیستند. آستانه‌های rating_band همان مقدار ثبت‌شده در manifest هستند.

In [ ]:
REQUIRED = {TEXT_COLUMN, *TARGETS.keys()}
def inspect(path, expected):
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig").columns.tolist()
    missing_cols = REQUIRED - set(header)
    if missing_cols: raise ValueError(f"{path.name}: ستون‌های مفقود {missing_cols}")
    sample = pd.read_csv(path, usecols=list(REQUIRED), nrows=5000, encoding="utf-8-sig")
    row = {"file": path.name, "columns": len(header), "preview_rows": len(sample), "manifest_rows": expected,
           "size_MB": round(path.stat().st_size / 1024**2, 1)}
    for target, labels in TARGETS.items():
        found = set(sample[target].dropna().astype(str).str.strip())
        unknown = found - set(labels)
        if unknown: raise ValueError(f"برچسب ناشناخته در {target}: {unknown}")
        row[target] = sorted(found)
    return row
checks = [inspect(TRAIN_PATH, EXPECTED_TRAIN_ROWS),
          inspect(VALID_PATH, manifest["output_row_counts"]["validation"]),
          inspect(TEST_PATH, manifest["output_row_counts"]["test"])]
display(pd.DataFrame(checks))
print("Rating bands:", manifest["rating_band_thresholds"])
print("Split:", manifest["split_strategy"]["method"])

## 5. نمونهٔ پایش ثابت validation

این نمونه یک‌بار به‌صورت chunkی ساخته و cache می‌شود. برای انتخاب مدل و early stopping است؛ آموزش نمی‌بیند. ارزیابی نهایی در ادامه روی همهٔ ردیف‌های validation و test خواهد بود.

In [ ]:
MONITOR_PATH = CACHE_DIR / f"validation_monitor_{VALIDATION_MONITOR_SIZE}_seed{VALIDATION_MONITOR_SEED}.csv"
if not MONITOR_PATH.exists():
    total = int(manifest["output_row_counts"]["validation"])
    fraction = min(1.0, VALIDATION_MONITOR_SIZE / total * 1.25)
    parts = []
    for i, chunk in enumerate(pd.read_csv(VALID_PATH, usecols=[TEXT_COLUMN, *TARGETS], chunksize=100_000, encoding="utf-8-sig")):
        parts.append(chunk.sample(frac=fraction, random_state=VALIDATION_MONITOR_SEED + i))
    monitor_frame = pd.concat(parts, ignore_index=True)
    if len(monitor_frame) > VALIDATION_MONITOR_SIZE:
        monitor_frame = monitor_frame.sample(n=VALIDATION_MONITOR_SIZE, random_state=VALIDATION_MONITOR_SEED)
    monitor_frame.to_csv(MONITOR_PATH, index=False, encoding="utf-8")
    del parts, monitor_frame; gc.collect()
monitor_frame = pd.read_csv(MONITOR_PATH, encoding="utf-8")
print("Validation monitor rows:", f"{len(monitor_frame):,}")

## 6. Tokenizer و معماری دوهدی

In [ ]:
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch import nn
from torch.utils.data import IterableDataset, Dataset, DataLoader
from safetensors.torch import save_file, load_file
TOKENIZER = AutoTokenizer.from_pretrained(CHECKPOINT, use_fast=True)

class TwoHeadParsBERT(nn.Module):
    def __init__(self, checkpoint, class_counts):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint)
        self.dropout = nn.Dropout(0.1)
        h = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({name: nn.Linear(h, n) for name, n in class_counts.items()})
        if USE_GRADIENT_CHECKPOINTING:
            self.encoder.gradient_checkpointing_enable()
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        pooled = self.dropout(out.last_hidden_state[:, 0])
        return {name: head(pooled) for name, head in self.heads.items()}

model = TwoHeadParsBERT(CHECKPOINT, {k: len(v) for k, v in TARGETS.items()}).to(DEVICE)
for p in model.parameters(): p.requires_grad = True
TOTAL_PARAMS = sum(p.numel() for p in model.parameters())
TRAINABLE_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters {TOTAL_PARAMS:,} | trainable {TRAINABLE_PARAMS:,}")
print("Tokenizer:", TOKENIZER.name_or_path, "| max_length:", MAX_LENGTH)

## 7. داده‌خوانی جریانی و batchسازی پویا

chunkها در هر epoch با seed ثابت shuffle می‌شوند. متن‌ها در هر batch tokenize و تا طول همان batch padding می‌شوند؛ این کار مصرف VRAM را نسبت به padding ثابت کاهش می‌دهد.

In [ ]:
LABEL_TO_ID = {t: {label: i for i, label in enumerate(labels)} for t, labels in TARGETS.items()}
IGNORE_INDEX = -100
class CSVReviewStream(IterableDataset):
    def __init__(self, path, training=False, row_indices=None):
        self.path, self.training, self.row_indices, self.epoch = str(path), training, row_indices, 0
    def __iter__(self):
        raw_offset = 0
        reader = pd.read_csv(self.path, usecols=[TEXT_COLUMN, *TARGETS], chunksize=CHUNK_ROWS, encoding="utf-8-sig")
        target_names = list(TARGETS)
        for chunk_id, frame in enumerate(reader):
            raw_count = len(frame)
            if self.training and self.row_indices is not None:
                left = np.searchsorted(self.row_indices, raw_offset, side="left")
                right = np.searchsorted(self.row_indices, raw_offset + raw_count, side="left")
                frame = frame.iloc[self.row_indices[left:right] - raw_offset]
            raw_offset += raw_count
            if frame.empty:
                continue
            if self.training:
                frame = frame.sample(frac=1, random_state=SEED + self.epoch * 1_000_003 + chunk_id)
            frame[TEXT_COLUMN] = frame[TEXT_COLUMN].fillna("").astype(str)
            for t in target_names:
                frame[t] = frame[t].map(LABEL_TO_ID[t]).fillna(IGNORE_INDEX).astype("int64")
            for row in frame.itertuples(index=False, name=None):
                yield {TEXT_COLUMN: row[0], target_names[0]: row[1], target_names[1]: row[2]}
class FrameDataset(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i): return self.frame.iloc[i].to_dict()
def collate(rows):
    batch = TOKENIZER([str(r[TEXT_COLUMN]) for r in rows], truncation=True, max_length=MAX_LENGTH,
                      padding=True, return_tensors="pt")
    for t in TARGETS: batch[t] = torch.tensor([int(r[t]) for r in rows], dtype=torch.long)
    return batch
def to_device(batch): return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
train_stream = CSVReviewStream(TRAIN_PATH, training=True, row_indices=TRAIN_ROW_INDICES)
monitor_loader = DataLoader(FrameDataset(monitor_frame), batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=0, collate_fn=collate, pin_memory=True)

## 8. تنظیم optimizer و بازیابی آموزش

نرخ یادگیری encoder از headها کمتر است. Drive نگه‌دارندهٔ یک checkpoint بازیابی rolling خواهد بود؛ فایل ممکن است بزرگ باشد، پس فضای کافی لازم است. پس از قطع runtime، این سلول وضعیت optimizer، scheduler، scaler و تاریخچه را بارگذاری می‌کند.

In [ ]:
encoder_params = list(model.encoder.parameters())
head_params = list(model.heads.parameters()) + list(model.dropout.parameters())
optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": ENCODER_LR},
                               {"params": head_params, "lr": HEAD_LR}], weight_decay=WEIGHT_DECAY)
total_steps = OPTIMIZER_STEPS_PER_EPOCH * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=True)
RECOVERY_PATH = CHECKPOINT_DIR / "latest_training_state.pt"
BEST_PATH = CHECKPOINT_DIR / "best_model.safetensors"
HISTORY_PATH = REPORT_DIR / "training_history.json"
CONFIG_PATH = REPORT_DIR / "run_config.json"
run_config = {"checkpoint": CHECKPOINT, "seed": SEED, "train_rows": TRAIN_ROWS,
 "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "gradient_accumulation": GRAD_ACCUMULATION,
 "effective_batch": EFFECTIVE_BATCH, "epochs": NUM_EPOCHS,
 "train_sampling": "seeded uniform sample by source row index", "train_sample_seed": TRAIN_SAMPLE_SEED, "encoder_lr": ENCODER_LR, "head_lr": HEAD_LR,
 "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP_RATIO, "validation_monitor_size": VALIDATION_MONITOR_SIZE,
 "validation_monitor_seed": VALIDATION_MONITOR_SEED, "gpu": GPU_NAME, "vram_gb": round(VRAM_GB, 2),
 "torch": torch.__version__, "transformers": transformers.__version__,
 "split_seed": manifest.get("split_seed"), "split_strategy": manifest.get("split_strategy", {}).get("method"),
 "rating_band_thresholds": manifest.get("rating_band_thresholds")}
CONFIG_PATH.write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8")
history = {"train_loss": [], "validation": [], "step_loss": []}
completed_epoch, resume_batch, global_step, best_score, bad_epochs = 0, 0, 0, -1.0, 0
if RECOVERY_PATH.exists():
    s = torch.load(RECOVERY_PATH, map_location="cpu", weights_only=False)
    model.load_state_dict(s["model"]); optimizer.load_state_dict(s["optimizer"])
    scheduler.load_state_dict(s["scheduler"]); scaler.load_state_dict(s["scaler"])
    history, completed_epoch, resume_batch = s["history"], s["completed_epoch"], s["batch_in_epoch"]
    global_step, best_score, bad_epochs = s["global_step"], s["best_score"], s["bad_epochs"]
    print("Resuming at epoch", completed_epoch + 1, "batch", resume_batch, "step", global_step)
else: print("Fresh run")
def save_recovery(epoch_idx, batch_idx):
    state = {"model": {k: v.detach().cpu() for k, v in model.state_dict().items()},
             "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
             "history": history, "completed_epoch": epoch_idx, "batch_in_epoch": batch_idx,
             "global_step": global_step, "best_score": best_score, "bad_epochs": bad_epochs}
    tmp = RECOVERY_PATH.with_suffix(".tmp")
    torch.save(state, tmp); os.replace(tmp, RECOVERY_PATH)
    del state

## 9. ارزیابی و نمودارهای پایش

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             balanced_accuracy_score, classification_report, confusion_matrix)
sns.set_theme(style="whitegrid", context="talk", palette="deep")

def evaluate(eval_model, loader, name):
    eval_model.eval()
    data = {t: {"y": [], "pred": []} for t in TARGETS}
    losses = []
    with torch.inference_mode():
        for batch in tqdm(loader, desc=name, leave=False):
            batch = to_device(batch)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = eval_model(batch["input_ids"], batch["attention_mask"])
                batch_losses = []
                for t in TARGETS:
                    y = batch[t]; mask = y != IGNORE_INDEX
                    if mask.any(): batch_losses.append(nn.functional.cross_entropy(logits[t][mask].float(), y[mask]))
                    data[t]["y"].extend(y[mask].cpu().tolist())
                    data[t]["pred"].extend(logits[t][mask].argmax(-1).cpu().tolist())
                if batch_losses: losses.append(float(torch.stack(batch_losses).mean().cpu()))
    result = {"split": name, "loss": float(np.mean(losses)) if losses else None}
    for t, v in data.items():
        y, pred = v["y"], v["pred"]
        if not y: continue
        result[t] = {"n": len(y), "accuracy": accuracy_score(y, pred),
          "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
          "macro_precision": precision_score(y, pred, average="macro", zero_division=0),
          "macro_recall": recall_score(y, pred, average="macro", zero_division=0),
          "balanced_accuracy": balanced_accuracy_score(y, pred), "y_true": y, "y_pred": pred}
    return result

def plot_history(hist):
    if not hist["train_loss"]: return
    fig, ax = plt.subplots(1, 2, figsize=(16, 6), dpi=150)
    ax[0].plot([r["epoch"] for r in hist["train_loss"]], [r["loss"] for r in hist["train_loss"]], marker="o", label="Train")
    vr = [r for r in hist["validation"] if r.get("loss") is not None]
    if vr: ax[0].plot([r["epoch"] for r in vr], [r["loss"] for r in vr], marker="s", label="Validation monitor")
    ax[0].set(title="Loss by epoch", xlabel="Epoch", ylabel="Cross-entropy"); ax[0].legend()
    for t in TARGETS:
        rows = [r for r in hist["validation"] if t in r]
        ax[1].plot([r["epoch"] for r in rows], [r[t]["macro_f1"] for r in rows], marker="o", label=t)
    ax[1].set(title="Validation monitor Macro F1", xlabel="Epoch", ylabel="Macro F1", ylim=(0, 1)); ax[1].legend()
    fig.tight_layout(); fig.savefig(PLOT_DIR / "training_curves.png", dpi=300, bbox_inches="tight"); plt.show(); plt.close(fig)

## 10. Fine-tuning روی نمونهٔ ثابت ۲ میلیونی

هر epoch دقیقاً همان ۲ میلیون ردیف را با همان seed می‌بیند. این حجم همچنان زمان‌بر است و احتمالاً به چند نشست رایگان نیاز دارد. checkpointها در Drive نگهداری می‌شوند. اگر نشست قطع شد، از ابتدای نوت‌بوک اجرا کنید تا وضعیت بازیابی شود. برآورد زمان واقعی پس از 200 batch نمایش داده می‌شود.

In [ ]:
optimizer.zero_grad(set_to_none=True)
for epoch_idx in range(completed_epoch, NUM_EPOCHS):
    train_stream.epoch = epoch_idx
    loader = DataLoader(train_stream, batch_size=BATCH_SIZE, num_workers=0, collate_fn=collate, pin_memory=True)
    model.train(); loss_sum = 0.0; loss_count = 0; pending = 0
    epoch_start_time = time.time()
    seen_batches = 0
    bar = tqdm(loader, total=MICRO_BATCHES_PER_EPOCH, desc=f"Epoch {epoch_idx+1}/{NUM_EPOCHS}")
    for batch_idx, batch in enumerate(bar):
        if epoch_idx == completed_epoch and batch_idx < resume_batch:
            seen_batches = batch_idx + 1; continue
        seen_batches = batch_idx + 1
        batch = to_device(batch)
        labels = {t: batch.pop(t) for t in TARGETS}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(batch["input_ids"], batch["attention_mask"])
            terms = []
            for t in TARGETS:
                mask = labels[t] != IGNORE_INDEX
                if mask.any(): terms.append(nn.functional.cross_entropy(logits[t][mask].float(), labels[t][mask]))
            if not terms: continue
            loss = torch.stack(terms).sum()
        scaler.scale(loss / GRAD_ACCUMULATION).backward()
        pending += 1; loss_sum += float(loss.detach().cpu()); loss_count += 1
        update = pending == GRAD_ACCUMULATION or batch_idx + 1 == MICRO_BATCHES_PER_EPOCH
        if update:
            scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1; pending = 0
            if global_step % 100 == 0: history["step_loss"].append({"step": global_step, "loss": loss_sum / max(loss_count, 1)})
            if global_step % SAVE_EVERY_OPTIMIZER_STEPS == 0: save_recovery(epoch_idx, seen_batches)
        bar.set_postfix(loss=f"{loss_sum/max(loss_count,1):.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}", step=global_step)
        if batch_idx == 199:
            elapsed = max(time.time() - epoch_start_time, 1e-6)
            projected_hours = elapsed / (batch_idx + 1) * MICRO_BATCHES_PER_EPOCH / 3600
            print(f"Estimated training time per epoch at current speed: {projected_hours:.2f} h (training only; excludes evaluation/checkpoint I/O)")
    resume_batch = 0
    train_loss = loss_sum / max(loss_count, 1)
    history["train_loss"].append({"epoch": epoch_idx + 1, "loss": train_loss})
    val = evaluate(model, monitor_loader, f"validation monitor epoch {epoch_idx+1}")
    score = float(np.mean([val[t]["macro_f1"] for t in TARGETS]))
    compact_val = {"split": val["split"], "loss": val["loss"]}
    for target in TARGETS:
        compact_val[target] = {k: v for k, v in val[target].items() if k not in {"y_true", "y_pred"}}
    history["validation"].append(compact_val)
    print("Train loss:", round(train_loss, 5), "| validation monitor mean Macro F1:", round(score, 5))
    if score > best_score:
        best_score, bad_epochs = score, 0
        save_file({k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()}, str(BEST_PATH))
        print("Best weights saved.")
    else: bad_epochs += 1
    completed_epoch = epoch_idx + 1
    HISTORY_PATH.write_text(json.dumps(history, ensure_ascii=False), encoding="utf-8")
    plot_history(history)
    save_recovery(completed_epoch, 0)
    del loader, bar; gc.collect(); torch.cuda.empty_cache()
    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print("Early stopping."); break
print("Training phase complete. Recovery:", RECOVERY_PATH)

## 11. بازگرداندن بهترین epoch و نمودارهای آموزش

In [ ]:
if not BEST_PATH.exists(): raise FileNotFoundError("Best checkpoint نیست؛ آموزش را اجرا کنید.")
model.load_state_dict(load_file(str(BEST_PATH), device="cpu")); model.to(DEVICE).eval()
plot_history(history)
print("Best validation monitor mean Macro F1:", round(best_score, 5))

## 12. ارزیابی نهایی روی splitهای کامل

هر دو split به‌طور جریانی و کامل ارزیابی می‌شوند؛ زمان‌بر است. از test برای تغییر تنظیمات یا انتخاب checkpoint استفاده نکنید.

In [ ]:
def full_loader(path):
    return DataLoader(CSVReviewStream(path, training=False), batch_size=BATCH_SIZE, num_workers=0,
                      collate_fn=collate, pin_memory=True)
full_valid = evaluate(model, full_loader(VALID_PATH), "validation-full")
full_test = evaluate(model, full_loader(TEST_PATH), "test-full")

def metrics_rows(result):
    rows=[]
    for t in TARGETS:
        d=result[t]
        rows.append({"split":result["split"],"target":t,"n":d["n"],"Accuracy":d["accuracy"],
          "Macro F1":d["macro_f1"],"Macro Precision":d["macro_precision"],
          "Macro Recall":d["macro_recall"],"Balanced Accuracy":d["balanced_accuracy"]})
    return rows
metrics = pd.DataFrame(metrics_rows(full_valid)+metrics_rows(full_test))
display(metrics.round(4)); metrics.to_csv(REPORT_DIR/"final_metrics.csv",index=False,encoding="utf-8-sig")
for result in [full_valid,full_test]:
    print("\n",result["split"])
    for t, labels in TARGETS.items():
        d=result[t]
        print("\n",t,"\n",classification_report(d["y_true"],d["y_pred"],labels=list(range(len(labels))),target_names=labels,digits=4,zero_division=0))
        if t=="rating_band":
            dist=np.abs(np.asarray(d["y_true"])-np.asarray(d["y_pred"]))
            print("Ordinal distance MAE:",round(float(dist.mean()),4))

### Confusion matrices

In [ ]:
def plot_confusions(result, tag):
    fig, axes = plt.subplots(1,2,figsize=(15,6),dpi=150)
    for ax,t in zip(axes,TARGETS):
        d=result[t]; cm=confusion_matrix(d["y_true"],d["y_pred"],labels=list(range(len(TARGETS[t]))),normalize="true")
        sns.heatmap(cm,annot=True,fmt=".2f",cmap="Blues",vmin=0,vmax=1,xticklabels=TARGETS[t],yticklabels=TARGETS[t],ax=ax)
        ax.set(title=t,xlabel="Predicted",ylabel="True")
    fig.tight_layout(); fig.savefig(PLOT_DIR/f"confusion_{tag}.png",dpi=300,bbox_inches="tight"); plt.show(); plt.close(fig)
plot_confusions(full_valid,"validation_full")
plot_confusions(full_test,"test_full")

## 13. ذخیرهٔ مدل و گزارش اجرا

فایل safetensors شامل وزن‌های encoder و headهاست. برای inference، معماری TwoHeadParsBERT بالا با checkpoint پایه ساخته و سپس همین وزن‌ها بارگذاری می‌شوند.

In [ ]:
ARTIFACT_DIR=OUTPUT_DIR/"artifacts"; ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
TOKENIZER.save_pretrained(ARTIFACT_DIR/"tokenizer")
model.encoder.config.save_pretrained(ARTIFACT_DIR/"encoder_config")
shutil.copy2(BEST_PATH,ARTIFACT_DIR/"model.safetensors")
metadata={"base_checkpoint":CHECKPOINT,"input":"body","max_length":MAX_LENGTH,"targets":TARGETS,
 "rating_band_thresholds":manifest.get("rating_band_thresholds"),"training_config":run_config,
 "best_validation_monitor_mean_macro_f1":best_score,"metrics":metrics.to_dict(orient="records"),
 "weight_file":"model.safetensors","tokenizer_dir":"tokenizer","encoder_config_dir":"encoder_config","model_name":"ParsBERT","classifier_dropout":0.1}
(ARTIFACT_DIR/"model_metadata.json").write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding="utf-8")
print("Artifacts:",ARTIFACT_DIR,"\nReports:",REPORT_DIR,"\nPlots:",PLOT_DIR)

## 14. موارد ثبت در گزارش

- تعداد ردیف‌های train استفاده‌شده، seed و روش نمونه‌گیری یکنواخت؛
- checkpoint، tokenizer، نسخهٔ کتابخانه‌ها، GPU/VRAM و مجوز مدل؛
- طول ورودی، batch مؤثر، نرخ یادگیری encoder و head، تعداد epoch و زمان اجرا؛
- پارامترهای کل و trainable، نمودارهای loss و Macro F1 و epoch منتخب؛
- معیارهای کامل validation/test برای هر دو هدف و confusion matrix؛
- ordinal distance برای rating_band؛
- هرگونه بازیابی از checkpoint، توقف، OOM یا تغییر تنظیمات؛
- عدم استفاده از artificial-unlabeled در این آزمایش نظارت‌شده.

خروجی‌ها در outputs/parsbert_finetuning هستند. checkpointهای حجیم را تا زمانی که مدل نهایی و گزارش را تأیید نکرده‌اید پاک نکنید.